In [26]:
import pandas as pd
import json

from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR,NuSVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import pandas as pd
from helpers.modeling import (
    identify_column_types,
    create_preprocessor,
    evaluate_model,
)

from tqdm import tqdm


with open('results.json', 'r') as f:
    results = json.load(f)

In [27]:
df = pd.read_csv("../Datasets/processed/UHPC_dataset/semantic_recoding_features_50_with_publications.csv")
df.head()
df['paper_reference'].nunique()
df['paper_reference'].value_counts().describe()

count    165.000000
mean      12.563636
std       15.086454
min        1.000000
25%        4.000000
50%        8.000000
75%       16.000000
max      112.000000
Name: count, dtype: float64

In [28]:
def run_pipeline(model_cls, model_key, kernel_kwargs=None):
    params = results["best_params"]["best_params_publications_included"][model_key]
    pipeline = Pipeline([('preprocessor', preprocessor),
                          ('model', model_cls(**(kernel_kwargs or {})))])
    pipeline.set_params(**params)
    pipeline.fit(X_train, y_train)

    train_metrics = evaluate_model(y_train, pipeline.predict(X_train))
    test_metrics = evaluate_model(y_test, pipeline.predict(X_test))
    return pipeline, train_metrics, test_metrics

In [29]:
X = df.drop(columns=["cs_28d", "paper_reference"])
y = df["cs_28d"]

pub_col = "paper_reference" 

numerical_cols, one_hot_columns, k_fold_columns = identify_column_types(X)

preprocessor = create_preprocessor(numerical_cols, one_hot_columns, k_fold_columns, 
                                   handle_unknown='ignore') 


In [30]:
model_configs = [
    (KNeighborsRegressor, 'knn', None),
    (SVR, 'svr', {'kernel': 'rbf'}),
    (NuSVR, 'nusvr', {'kernel': 'rbf'}),
]

groups = list(df.groupby(pub_col))

lopo_results = []
lopo_predictions = []  

for pub_id, group_df in tqdm(groups, total=len(groups)):
    if len(group_df) < 5:
        continue

    train_mask = df[pub_col] != pub_id
    test_mask  = df[pub_col] == pub_id

    X_train, y_train = X[train_mask], y[train_mask]
    X_test,  y_test  = X[test_mask],  y[test_mask]

    for model_cls, model_key, kwargs in model_configs:
        pipeline, train_metrics, test_metrics = run_pipeline(model_cls, model_key, kwargs)
        test_metrics.update({'publication': pub_id, 'model': model_key, 'train_RMSE': train_metrics['RMSE']})
        lopo_results.append(test_metrics)

        y_pred = pipeline.predict(X_test)
        for true_val, pred_val, idx in zip(y_test.values, y_pred, y_test.index):
            lopo_predictions.append({'index': idx, 'publication': pub_id, 'model': model_key,
                                       'y_true': true_val, 'y_pred': pred_val})

lopo_df = pd.DataFrame(lopo_results)
lopo_pred_df = pd.DataFrame(lopo_predictions)

  0%|          | 0/165 [00:00<?, ?it/s]

100%|██████████| 165/165 [03:07<00:00,  1.14s/it]


In [ ]:
# Pooled LOPO summary:
# Every row in the dataset was held out exactly once (by the model that never
# saw its publication during training). This pools ALL those held-out
# predictions together (across all 165 eligible publications) and computes
# ONE overall RMSE/MAE/R2/Correlation/Mean_Residual per model - as if it were
# a single big "unseen publication" test set.

pooled_summary = []
for model_key, group in lopo_pred_df.groupby('model'):
    m = evaluate_model(group['y_true'], group['y_pred'])
    m['model'] = model_key
    m['N_total'] = len(group)
    pooled_summary.append(m)

pooled_lopo_df = pd.DataFrame(pooled_summary)
pooled_lopo_df

,RMSE,MAE,R2,Correlation,Mean_Residual,N,model,N_total
0,24.782641,19.009600,0.534245,0.731417,-0.140068,1947,knn,1947
1,26.896383,21.386751,0.451407,0.673645,-1.755926,1947,nusvr,1947
2,26.891412,21.270024,0.451609,0.672979,0.033511,1947,svr,1947


In [37]:
lopo_pred_df['residual'] = lopo_pred_df['y_true'] - lopo_pred_df['y_pred']

residual_pivot = lopo_pred_df.pivot(index='index', columns='model', values='residual')
residual_pivot['mean_abs_residual'] = residual_pivot[['knn', 'svr', 'nusvr']].abs().mean(axis=1)
residual_pivot['same_direction'] = residual_pivot[['knn', 'svr', 'nusvr']].apply(
    lambda r: (r > 0).all() or (r < 0).all(), axis=1
)

worst_rows = residual_pivot.sort_values('mean_abs_residual', ascending=False).head(10)
worst_rows

model,knn,nusvr,svr,mean_abs_residual,same_direction
index,,,,,
1579,118.823380,121.468788,123.923303,121.405157,True
1580,100.023828,89.634561,91.880814,93.846401,True
1578,90.022758,94.349726,96.692005,93.688163,True
1575,72.142272,87.107076,90.004700,83.084683,True
1574,66.931021,86.214977,88.805532,80.650510,True
1214,99.990827,69.201323,72.687276,80.626475,True
710,-88.319086,-72.098183,-76.803632,79.073634,True
1427,-68.169831,-81.625912,-83.810911,77.868885,True
805,58.827838,87.343197,85.388436,77.186490,True


In [39]:
worst_idx = worst_rows.head(10).index 

df.loc[worst_idx].T

index,1579,1580,1578,1575,1574,1214,710,1427,805,813
cement,991.74,1061.95,975.61,941.18,948.62,816.0,170.0,526.1,752.0,752.0
cement_type,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,OPC_42.5,OPC_42.5,OPC_42.5,OPC_42.5
silica_fume,247.93,265.49,243.9,235.29,237.15,144.0,260.0,210.5,282.0,282.0
fly_ash,0.0,0.0,0.0,0.0,0.0,0.0,680.0,0.0,0.0,0.0
fly_ash_type,not_applicable,not_applicable,not_applicable,not_applicable,not_applicable,not_applicable,class C,not_applicable,not_applicable,not_applicable
limestone_powder,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
quartz_powder,0.0,0.0,0.0,0.0,0.0,0.0,0.0,259.6,0.0,0.0
glass_powder,247.93,265.49,243.9,235.29,237.15,240.0,0.0,0.0,0.0,0.0
rice_husk_ash,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
metakaolin,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [32]:
lopo_df.to_csv("results/lopo_results.csv", index=False)

In [33]:
summary = lopo_df.groupby('model')[['RMSE', 'MAE', 'R2', 'Correlation', 'Mean_Residual']].agg(['mean', 'median', 'std'])
summary

RMSE                              MAE                        \
            mean     median        std       mean     median        std   
model                                                                     
knn    21.419475  19.572184  11.775280  18.604688  16.208164  10.873616   
nusvr  23.171636  19.937524  12.997444  20.766042  17.842043  12.608821   
svr    23.135482  19.538538  13.156764  20.664893  17.172159  12.859098   

             R2                      Correlation                      \
           mean    median        std        mean    median       std   
model                                                                  
knn   -6.613961 -0.592579  33.570450    0.428162  0.592603  0.525511   
nusvr -7.219633 -0.848165  23.358945    0.537490  0.760083  0.510762   
svr   -7.499590 -0.887777  24.930110    0.534684  0.726080  0.487200   

      Mean_Residual                       
               mean    median        std  
model                                     
knn       -0.823421 -2.136538  19.169234  
nusvr     -1.928766 -3.642213  22.334722  
svr        0.131418 -1.884150  22.476054

In [34]:
per_pub = lopo_df[['publication', 'model', 'N', 'RMSE', 'MAE', 'R2', 'Correlation', 'Mean_Residual']].sort_values(['model', 'publication'])
per_pub.to_csv("results/lopo_per_publication.csv", index=False)

In [35]:
worst = lopo_df.sort_values('R2').groupby('model').head(5)[['model', 'publication', 'N', 'RMSE', 'R2', 'Correlation', 'Mean_Residual']]
worst

,model,publication,N,RMSE,R2,Correlation,Mean_Residual
321,knn,Ref-75-Research,5,20.708217,-362.452426,0.490140,20.686111
323,nusvr,Ref-75-Research,5,15.604997,-205.390432,0.296691,15.546208
322,svr,Ref-75-Research,5,14.689291,-181.879004,0.401032,14.631977
253,svr,Ref-52-Research,6,30.615777,-160.453251,-0.350338,30.412859
254,nusvr,Ref-52-Research,6,26.511428,-120.066075,-0.350812,26.365055
73,svr,Ref-130-Research,6,29.275453,-112.433370,-0.356493,29.035872
37,svr,Ref-114-Research,5,44.170918,-80.403311,-0.717704,41.967343
74,nusvr,Ref-130-Research,6,24.791305,-80.345285,-0.373915,24.617829
210,knn,Ref-37-Research,7,27.782989,-71.308644,0.733811,-27.667405
38,nusvr,Ref-114-Research,5,39.302746,-63.448826,-0.701915,37.014362


In [36]:
lopo_df['direction'] = lopo_df['Mean_Residual'].apply(lambda x: 'under-predicted' if x > 0 else 'over-predicted')
direction_summary = lopo_df.groupby(['model', 'direction']).size().unstack(fill_value=0)
direction_summary

direction,over-predicted,under-predicted
model,,
knn,65,58
nusvr,67,56
svr,64,59
